# KrishiInsights — Agro-Climatic Correlation & Outlier Analysis

**Project:** KrishiInsights  
**Overlapping Window:** 1997–2015 (19 common harvest years)  
**Database:** Neon Serverless PostgreSQL (`neondb`)  
**Stage:** 4 (Statistical Analysis & Exploratory Data Analysis)

### Objectives
1. Measure paired seasonal statistical relationships (Pearson $r$, Spearman $\rho$, and $p$-values).
2. Evaluate crop-specific rainfall sensitivities (rainfed vs irrigated crops).
3. Demonstrate state-level sensitivity divergence (Punjab/Haryana irrigation vs Odisha/Bihar rainfed).
4. Perform IQR and Z-score outlier detection and classify agro-climatic extremes vs data anomalies.
5. Emphasize critical agricultural caveats (correlation $\neq$ causation).


In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath("../.."))

import pandas as pd
from ml.src.analytics.correlation_analysis import (
    calculate_seasonal_correlations,
    calculate_crop_rainfall_correlations,
    calculate_state_level_rainfall_sensitivity
)
from ml.src.analytics.outlier_analysis import (
    detect_outliers_iqr,
    detect_outliers_zscore,
    analyze_crop_yield_anomalies,
    classify_anomalies_summary
)
from ml.src.database.connection import load_agricultural_data

print("Correlation and outlier modules successfully imported.")


Correlation and outlier modules successfully imported.


## 1. Paired Seasonal Correlation Analysis (1997–2015)
Aligns Southwest Monsoon with Kharif, Winter rainfall with Rabi, and Pre-monsoon with Summer.


In [2]:
seasonal_corr_df = calculate_seasonal_correlations()
print("=== Paired Seasonal Correlations ===")
print(seasonal_corr_df.to_string(index=False))


=== Paired Seasonal Correlations ===
                             relationship     season  sample_size  pearson_r  pearson_p_value  spearman_rho  spearman_p_value  significance_at_5pct
         Monsoon Rainfall vs Kharif Yield     Kharif          492    -0.1364     2.422800e-03       -0.0801      7.582200e-02                  True
    Monsoon Rainfall vs Kharif Production     Kharif          492    -0.1551     5.566500e-04       -0.2935      3.148700e-11                  True
              Rabi Rainfall vs Rabi Yield       Rabi          470    -0.2273     6.374400e-07       -0.3135      3.555500e-12                  True
         Rabi Rainfall vs Rabi Production       Rabi          470    -0.3298     2.173500e-13       -0.3992      2.076000e-19                  True
     Pre-Monsoon Rainfall vs Summer Yield     Summer          185     0.3103     1.718500e-05        0.4421      2.971500e-10                  True
Pre-Monsoon Rainfall vs Summer Production     Summer          185     0.195

## 2. Crop-Specific Rainfall Correlations
Contrasts rainfed coarse grains and pulses against irrigated wheat and sugarcane.


In [3]:
crop_corr_df = calculate_crop_rainfall_correlations()
print("=== Crop-Specific Correlation with Annual Rainfall ===")
print(crop_corr_df.to_string(index=False))


=== Crop-Specific Correlation with Annual Rainfall ===
   crop_name crop_category aligned_rainfall  sample_size  yield_pearson_r  yield_pearson_p  yield_spearman_rho  prod_pearson_r  statistically_significant
   Groundnut       Oilseed          Monsoon          548           0.0630     1.406700e-01             -0.0593         -0.1833                      False
   Arhar/Tur         Pulse           Annual          400           0.0487     3.310300e-01             -0.0308         -0.0358                      False
       Jowar Coarse Cereal          Monsoon          418          -0.0278     5.711000e-01             -0.0020          0.2296                      False
Cotton(lint)         Fiber          Monsoon          362          -0.0556     2.914600e-01             -0.1814         -0.1065                      False
        Gram         Pulse    Rabi / Winter          380          -0.1137     2.663900e-02             -0.1971         -0.2157                       True
        Rice        C

## 3. State-Level Sensitivity Contrast: Rice (Kharif)
Contrasts high-irrigation canal states against rainfed eastern/central states.


In [4]:
state_sens_df = calculate_state_level_rainfall_sensitivity(target_crop="Rice")
print("=== State-Level Rainfall Sensitivity Contrast ===")
print(state_sens_df.to_string(index=False))


=== State-Level Rainfall Sensitivity Contrast ===
            state_name crop  years_analyzed  yield_pearson_r  yield_spearman_rho  p_value  mean_monsoon_rainfall  mean_yield
             Rajasthan Rice              14           0.7340              0.7407 0.002803                  368.3        1.44
        Madhya Pradesh Rice              15           0.7231              0.7286 0.002317                  937.9        1.00
          Chhattisgarh Rice              15           0.5881              0.6321 0.021111                 1129.4        1.39
             Telangana Rice              17           0.5668              0.5098 0.017662                  769.7        2.77
               Gujarat Rice              15           0.5429              0.4679 0.036494                  730.4        1.68
      Himachal Pradesh Rice              12           0.5277              0.3217 0.077835                  674.4        1.48
             Karnataka Rice              18           0.5235              0

## 4. Outlier Analysis and Anomaly Classification
Detects statistical outliers and separates legitimate agro-climatic extremes from data anomalies.


In [5]:
agri_df = load_agricultural_data()
macro_anomalies = classify_anomalies_summary(agri_df)

print("=== Macro Anomaly Classification Summary ===")
for k, v in macro_anomalies.items():
    print(f"  {k}: {v}")

print("\n=== Rice Yield Outlier Analysis ===")
rice_anomalies = analyze_crop_yield_anomalies(agri_df, crop_name="Rice")
print(f"Total Rice Records: {rice_anomalies['total_records']:,}")
print(f"IQR Upper Bound: {rice_anomalies['iqr_analysis']['upper_bound']} Tonnes/Ha")
print(f"Outliers Count: {rice_anomalies['iqr_analysis']['outlier_count']} ({rice_anomalies['iqr_analysis']['outlier_percentage']}%)")


=== Macro Anomaly Classification Summary ===
  field_crops_evaluated: 240338
  suspicious_small_area_high_production_count: 20
  severe_crop_failures_count: 878
  sugarcane_iqr_bounds: {'q1': 37.1535, 'q3': 70.5128, 'upper_bound': 120.5517, 'outliers_above_bound': 154}
  rice_iqr_bounds: {'q1': 1.2749, 'q3': 2.565, 'upper_bound': 4.5001, 'outliers_above_bound': 153}

=== Rice Yield Outlier Analysis ===
Total Rice Records: 15,081
IQR Upper Bound: 4.5001 Tonnes/Ha
Outliers Count: 153 (1.01%)


## 5. Agricultural Caveats & Methodological Warnings
> **Important Disclaimer:**  
> A statistical correlation between rainfall and crop yield does **not** imply direct causality.  
> Agricultural yield is heavily mediated by:
> 1. Irrigation infrastructure (canals, tube-wells)
> 2. Fertilizer and High-Yielding Variety (HYV) seed application
> 3. Soil health, water retention, and topography
> 4. Farm management practices and state procurement policies
